# Deploy WhisperLiveKit to Amazon SageMaker

This notebook builds a SageMaker-compatible container for WhisperLiveKit, pushes it to Amazon ECR,
and provisions a real-time endpoint that accepts `audio/webm` payloads from the default browser client.

## Prerequisites

* You are running inside an AWS environment that has access to SageMaker, Amazon ECR, and IAM.
* Docker is available in the notebook kernel (SageMaker Studio/Notebook Instance).
* Your execution role has permission to create ECR repositories and SageMaker models/endpoints.

In [ ]:
import boto3, json, os, time, sagemaker
from pathlib import Path

session = boto3.session.Session()
region = session.region_name or 'us-west-2'
account_id = boto3.client('sts').get_caller_identity()['Account']

try:
    role = sagemaker.get_execution_role()
except ValueError:
    raise ValueError('Set `role` to your SageMaker execution role ARN before continuing.')

image_name = 'whisperlivekit-sagemaker'
image_tag = 'latest'
repository_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{image_name}:{image_tag}"
model_name = f"whisperlivekit-model-{int(time.time())}"
endpoint_config_name = f"whisperlivekit-config-{int(time.time())}"
endpoint_name = f"whisperlivekit-endpoint-{int(time.time())}"
instance_type = 'ml.c5.4xlarge'  # adjust to match your capacity requirements

os.environ['REGION'] = region
os.environ['ACCOUNT_ID'] = account_id
os.environ['ECR_REPO'] = image_name
os.environ['IMAGE_URI'] = repository_uri

print('Region:', region)
print('Repository URI:', repository_uri)
print('Model name:', model_name)
print('Endpoint name:', endpoint_name)
print('Instance type:', instance_type)
print('Execution role:', role)


## Build and push the WhisperLiveKit container

This step logs in to Amazon ECR (creating the repository on first run), builds the Docker image
from `sagemaker/Dockerfile`, and pushes the tagged image to your registry.

In [ ]:
%%bash
set -eux
aws ecr describe-repositories --repository-names "${ECR_REPO}" --region "${REGION}" >/dev/null 2>&1 || \
  aws ecr create-repository --repository-name "${ECR_REPO}" --region "${REGION}"
aws ecr get-login-password --region "${REGION}" | docker login --username AWS --password-stdin "${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com"
docker build --build-arg REGION=${REGION} -f sagemaker/Dockerfile -t ${IMAGE_URI} .
docker push ${IMAGE_URI}


## Create the SageMaker model and endpoint

The environment variables configure WhisperLiveKit to keep FFmpeg enabled and stream `audio/webm` payloads.
`WHISPER_MODEL_SIZE` defaults to `tiny`; override it here if you need a larger checkpoint.

In [ ]:
sm_client = boto3.client('sagemaker', region_name=region)
container = {
    'Image': repository_uri,
    'Environment': {
        'WHISPER_MODEL_SIZE': 'base',
        'WHISPER_MIN_CHUNK_SIZE': '0.5',
        'WHISPER_VAC': 'true',
        'WHISPER_VAD': 'true',
    },
}

sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer=container,
)

sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[{
        'VariantName': 'AllTraffic',
        'ModelName': model_name,
        'InitialInstanceCount': 1,
        'InstanceType': instance_type,
        'InitialVariantWeight': 1.0,
    }],
)

sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

print('Waiting for endpoint to become InService...')
sm_client.get_waiter('endpoint_in_service').wait(EndpointName=endpoint_name)
print('Endpoint status:', sm_client.describe_endpoint(EndpointName=endpoint_name)['EndpointStatus'])


## Invoke the endpoint with `audio/webm`

Provide the path to an Opus-encoded WebM sample recorded by the default browser client.
The endpoint returns the same JSON structure as the WebSocket server.

In [ ]:
runtime = boto3.client('sagemaker-runtime', region_name=region)
sample_audio = Path('sample_audio.webm')  # update with your recording
if not sample_audio.exists():
    raise FileNotFoundError(f"Sample audio not found: {sample_audio}")

with sample_audio.open('rb') as f:
    audio_bytes = f.read()

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='audio/webm',
    Body=audio_bytes,
)
payload = json.loads(response['Body'].read().decode('utf-8'))
print(json.dumps(payload, indent=2))


## Clean up resources

Run the following cell when you are finished testing to avoid incurring ongoing charges.

In [ ]:
sm_client.delete_endpoint(EndpointName=endpoint_name)
sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm_client.delete_model(ModelName=model_name)
print('Cleanup complete.')
